# 3ptWL-mod: Four-Model Comparison in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadirs/3ptWL-mod/blob/main/examples/3ptWL_mod_four_models_colab.ipynb)

This standalone notebook compares the four 3PCF theory branches provided by **3ptWL-mod**:

1. standard perturbation theory (SPT);
2. tree/power-spectrum branch;
3. effective field theory (EFT);
4. Takahashi/Halo Model.

It does **not** clone the repository. It installs the published package from PyPI and downloads one tagged example power spectrum from GitHub. The main result is a shared-color-scale grid with one row per model and one column per multipole.

## 1. Install 3ptWL-mod

Run this cell once in a fresh Colab runtime. Native GSL and FFTW3 libraries are installed before pip compiles the Python extension. The default fast configuration below is intended for an interactive demonstration, not a production convergence study.

In [ ]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        [
            "apt-get", "-qq", "install", "-y",
            "build-essential", "libgsl-dev", "libfftw3-dev",
            "python3-dev", "pkg-config",
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "3ptWL-mod==1.0.0"],
    check=True,
)
print("3ptWL-mod and its Python dependencies are ready.")

## 2. Imports and example input

The example input is pinned to the same `v1.0.0` release as the Python package, making the notebook reproducible and independent of future changes on the default branch.

In [ ]:
from collections import OrderedDict
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlretrieve
import os

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
from wlcfpy import wlcf

WORK_DIR = Path.cwd() / "3ptwl_colab"
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "outputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PS_URL = (
    "https://raw.githubusercontent.com/sadirs/3ptWL-mod/"
    "v1.0.0/input/linear_pk_Takahashi_z0.txt"
)
PS_FILE = INPUT_DIR / "linear_pk_Takahashi_z0.txt"
if not PS_FILE.exists():
    urlretrieve(PS_URL, PS_FILE)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print(f"3ptWL-mod version: {version('3ptWL-mod')}")
print(f"Input: {PS_FILE}")
print(f"Outputs: {OUTPUT_DIR}")

## 3. Configure the four models

`FAST_MODE=True` uses a compact grid suitable for Colab. Set it to `False` for the higher-resolution settings used by the repository example. Always perform an independent convergence study before using these predictions in scientific inference.

In [ ]:
FAST_MODE = True

resolution = (
    {"Nell": 48, "chiQuadSteps": 80, "GLpoints": 24}
    if FAST_MODE
    else {"Nell": 64, "chiQuadSteps": 120, "GLpoints": 32}
)

base_params = {
    "fnamePS": str(PS_FILE.resolve()),
    "rootDir": str(OUTPUT_DIR.resolve()),
    "numberThreads": max(1, min(2, os.cpu_count() or 1)),
    "verbose": 0,
    "verbose_log": 0,
    "mMax": 4,
    "writevectors": 0,
    **resolution,
}

models = OrderedDict([
    ("SPT", {"tree_level": 1, "prefix": "colab_spt_"}),
    ("Tree", {"tree_level": 2, "prefix": "colab_tree_"}),
    ("EFT", {"tree_level": 3, "prefix": "colab_eft_"}),
    ("Halo Model", {"tree_level": 4, "prefix": "colab_halo_"}),
])

MOMENTS = tuple(range(5))
ARC_MIN_PER_RAD = 180.0 / np.pi * 60.0
PLOT_THETA_LIMITS = (10, 200)
COLOR_SCALE_MIN = 1e-11
COLOR_SCALE_MAX = 1e-8

base_params

## 4. Run all four branches

Each branch uses the same input spectrum and numerical resolution. Only `tree_level` and the output prefix change.

In [ ]:
def run_model(name, config):
    params = dict(
        base_params,
        prefix=config["prefix"],
        tree_level=config["tree_level"],
    )
    runner = wlcf()
    runner.set(params)
    cpu_time = runner.Run()
    runner.clean_all()
    print(
        f"{name:10s} | tree_level={config['tree_level']} | "
        f"CPU/thread={cpu_time:.3f} s"
    )
    return cpu_time


def load_model(prefix):
    theta = np.loadtxt(OUTPUT_DIR / f"{prefix}theta_array.txt") * ARC_MIN_PER_RAD
    zetas = {
        m: np.abs(np.loadtxt(OUTPUT_DIR / f"{prefix}zetam{m}.txt"))
        for m in MOMENTS
    }
    return theta, zetas


timings = {}
model_zetas = OrderedDict()
theta = None

for model_name, config in models.items():
    timings[model_name] = run_model(model_name, config)
    model_theta, model_zetas[model_name] = load_model(config["prefix"])
    if theta is None:
        theta = model_theta
    elif not np.allclose(theta, model_theta):
        raise RuntimeError(f"Theta grid differs for {model_name}.")

print(f"Theta range: {theta.min():.2f} to {theta.max():.2f} arcmin")
print(f"Map shape: {model_zetas['SPT'][0].shape}")

## 5. Compare the 3PCF multipole maps

Rows show theory branches and columns show multipoles. Every panel uses the same logarithmic color scale, so differences in amplitude can be compared directly. The absolute value is displayed because individual multipoles can change sign.

In [ ]:
Theta2, Theta1 = np.meshgrid(theta, theta)
shared_norm = LogNorm(vmin=COLOR_SCALE_MIN, vmax=COLOR_SCALE_MAX)
nrows = len(models)
ncols = len(MOMENTS)

fig = plt.figure(figsize=(20, 13), constrained_layout=False)
gs = fig.add_gridspec(
    nrows=nrows,
    ncols=ncols + 1,
    width_ratios=[1] * ncols + [0.05],
    wspace=0.20,
    hspace=0.25,
)

for row, (model_name, zetas) in enumerate(model_zetas.items()):
    for col, m in enumerate(MOMENTS):
        ax = fig.add_subplot(gs[row, col])
        mesh = ax.pcolormesh(
            Theta2, Theta1, zetas[m],
            shading="auto", cmap="RdYlBu_r", norm=shared_norm,
        )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(*PLOT_THETA_LIMITS)
        ax.set_ylim(*PLOT_THETA_LIMITS)
        ax.set_aspect("equal", adjustable="box")

        if row == 0:
            ax.set_title(fr"$m={m}$", fontsize=15)
        if col == 0:
            ax.set_ylabel(model_name + "\n" + r"$\theta_1$ [arcmin]")
        if row == nrows - 1:
            ax.set_xlabel(r"$\theta_2$ [arcmin]")

cax = fig.add_subplot(gs[:, -1])
cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label(r"$|\zeta_m|$", fontsize=15)
fig.suptitle("3ptWL-mod: Four-Model 3PCF Comparison", fontsize=19, y=0.995)

MAP_FIGURE = WORK_DIR / "four_models_3pcf_maps.png"
fig.savefig(MAP_FIGURE, bbox_inches="tight")
plt.show()
print(f"Saved: {MAP_FIGURE}")

## 6. Compare one-dimensional slices

These panels fix `theta_2` near 13, 72, and 169 arcmin and compare all four models while varying `theta_1`.

In [ ]:
theta2_targets = (13, 72, 169)
slice_moments = (0, 1, 2)
colors = {
    "SPT": "#0072B2",
    "Tree": "#009E73",
    "EFT": "#D55E00",
    "Halo Model": "#CC79A7",
}

fig, axes = plt.subplots(
    len(theta2_targets), len(slice_moments),
    figsize=(12, 10), sharex=True, sharey="col",
)

for row, theta2_target in enumerate(theta2_targets):
    index = int(np.argmin(np.abs(theta - theta2_target)))
    theta2_actual = theta[index]

    for col, m in enumerate(slice_moments):
        ax = axes[row, col]
        for model_name, zetas in model_zetas.items():
            ax.plot(
                theta, zetas[m][:, index],
                label=model_name, color=colors[model_name], lw=1.8,
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(*PLOT_THETA_LIMITS)
        ax.grid(alpha=0.20)

        if row == 0:
            ax.set_title(fr"$m={m}$")
        if col == 0:
            ax.set_ylabel(fr"$|\zeta_m|$ at $\theta_2={theta2_actual:.0f}'$")
        if row == len(theta2_targets) - 1:
            ax.set_xlabel(r"$\theta_1$ [arcmin]")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle("Four-Model Multipole Slices", fontsize=18, y=0.995)
fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.965),
    ncol=4, frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.90))

SLICE_FIGURE = WORK_DIR / "four_models_3pcf_slices.png"
fig.savefig(SLICE_FIGURE, bbox_inches="tight")
plt.show()
print(f"Saved: {SLICE_FIGURE}")

## 7. Package the generated results

This optional cell creates one zip archive containing the numerical outputs and both figures. In Colab, uncomment the final two lines to download it.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(WORK_DIR.parent / "3ptwl_four_models_results"),
    "zip",
    root_dir=WORK_DIR,
)
print(f"Created: {archive}")

# from google.colab import files
# files.download(archive)

## Scientific note

The fast settings are meant to expose the workflow and qualitative model differences. For publication-quality predictions, increase `Nell`, `chiQuadSteps`, and `GLpoints`, verify numerical convergence, and replace the example power spectrum with one generated for the intended cosmology and redshift.

If this modeling framework contributes to scientific work, cite Abraham Arvizu et al., *Modeling the 3-point correlation function of projected scalar fields on the sphere*, JCAP **12** (2024) 049, [arXiv:2408.16847](https://arxiv.org/abs/2408.16847).